In [9]:
import open3d as o3d
print(o3d.__version__)

0.16.1


In [10]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from rod.rod_generator import RodGenerator

from collections import defaultdict
import open3d as o3d

In [11]:
def plot_cls(poses):
    """
    Creates an interactive visualization of multiple centerlines.

    Parameters:
    - poses: [(n_sites, 3)] Centerline positions
    """
    # Create figure
    fig = go.Figure()

    # Plot each helix
    for idx, pos in enumerate(poses):
        # Plot centerline
        fig.add_trace(go.Scatter3d(
            x=pos[:, 0], y=pos[:, 1], z=pos[:, 2],
            mode='lines',
            line=dict(width=3),
            name=f'Centerline {idx + 1}'
        ))


    # Update layout
    fig.update_layout(
        scene=dict(
            xaxis=dict(title='X'),
            yaxis=dict(title='Y'),
            zaxis=dict(title='Z'),
            aspectmode='data'
        ),
        title=dict(text='CL Visualization', y=0.95, x=0.5, xanchor='center', yanchor='top'),
    )

    fig.show()

In [12]:
'''
Input: set of centerlines on a scalp

Algorithm:
- normalize relative positions of root point from each CL on the scalp on some surface
- compute negative density gradient for each CL root point on the surface
- calculate force on each CL point based on density gradient
   - force magnitude falls off with distance from root point, but has same directionality
   - root points DO NOT MOVE, but other points do
- update CL points based on force
- repeat until equilibrium configuration is reached (loss will be some "energy" related to the centerline force)

Output: set of CLs with updated positions
'''

# generating a set of CLs
poses = [] # n x 3
thetas = []
for i in range(10):
    pos, theta = RodGenerator.straight_rod(20)
    pos[:, 0] += 2 * 2 * np.random.sample() # spacing out x coords
    pos[:, 1] += 2 * 2 * np.random.sample() # spacing out y coords
    poses.append(pos)
    thetas.append(theta)

roots = np.array([pos[0] for pos in poses])
poses = np.array(poses)

plot_cls(poses)

In [13]:
# Inserting points from the centerlines into an Octree

# Flatten poses
num_strands, points_per_strand, _ = poses.shape
all_points = poses.reshape(-1, 3)

# Build map from point (as tuple) to strand ID
point_strand_map = {
    tuple(all_points[i]): strand_id
    for strand_id in range(num_strands)
    for i in range(strand_id * points_per_strand, (strand_id + 1) * points_per_strand)
}

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(all_points)
octree = o3d.geometry.Octree(max_depth=5)
octree.convert_from_point_cloud(pcd, size_expand=0.01)

In [14]:
# Octree functionality
def collect_neighbors(octree, query_point, radius):
    neighbors = []

    def callback(node, node_info):
        # Check if node is a leaf and contains children
        if isinstance(node, o3d.geometry.OctreeLeafNode):
            for child_point in node.indices:
                pt = np.asarray(pcd.points)[child_point]
                dist = np.linalg.norm(pt - query_point)
                if dist <= radius:
                    neighbors.append((child_point, pt))
        return False  # Don't terminate early

    octree.traverse(callback)
    return neighbors

def get_strandwise_neighbors(query_idx, radius=1.0):
    query_point = all_points[query_idx]
    current_strand = point_strand_map[tuple(query_point)]
    
    neighbors = collect_neighbors(octree, query_point, radius)
    other_strand_neighbors = [
        (idx, pt) for idx, pt in neighbors
        if point_strand_map.get(tuple(pt), -1) != current_strand
    ]
    return other_strand_neighbors

def compute_density_gradient(query_idx, radius=1.0):
    radial_kernel = lambda r: np.exp(-r**2 / (2 * radius**2))
    query_point = all_points[query_idx]
    neighbors = get_strandwise_neighbors(query_idx, radius)
    grad_d = np.zeros(3)

    for n in neighbors:
        mag = np.linalg.norm(n - query_point)
        if mag > 0:
            grad_d += radial_kernel(mag) * (query_point - n) / mag

    return grad_d

def compute_energy(query_idx, radius=1.0, alpha=2.0, flat_arc_lengths=None):
    # appying an energy decay based on arc length from root point
    density_grad = compute_density_gradient(query_idx, radius) # away from direction of high density
    s = flat_arc_lengths[query_idx]
    decay_weight = np.exp(-alpha * s)
    energy = np.linalg.norm(density_grad) * decay_weight
    return energy

def precompute_arc_lengths(poses):
    num_strands, num_points, _ = poses.shape
    arc_lengths = np.zeros((num_strands, num_points))
    for i in range(num_strands):
        diffs = np.linalg.norm(np.diff(poses[i], axis=0), axis=1)  # length between successive points
        arc_lengths[i, 1:] = np.cumsum(diffs)  # arc length from root
    flat_arc_lengths = arc_lengths.flatten()
    return flat_arc_lengths

# approximating force at each point
def compute_force(query_idx, radius=1.0, alpha=2.0, flat_arc_lengths=None):
    query_point = all_points[query_idx]
    s = flat_arc_lengths[query_idx]
    decay = np.exp(-alpha * s)

    neighbors = get_strandwise_neighbors(query_idx, radius)
    grad_rho = np.zeros(3)
    force = np.zeros(3)

    for idx, pt in neighbors:
        diff = query_point - pt
        mag = np.linalg.norm(diff)
        if mag > 1e-6:
            dir = diff / mag
            weight = np.exp(-mag**2 / (2 * radius**2))
            grad_rho += weight * dir
            # Force term: anti-parallel to gradient of gradient (Laplacian-like)
            force -= weight * dir / mag  # This acts like -∇|∇ρ|

    force *= decay
    return force

In [15]:
print(poses.shape)

(10, 22, 3)


In [16]:
from tqdm import tqdm
from cl_solver import CL_Simulator

sim = CL_Simulator(
    strands=poses,  # shape (N_strands, N_points, 3)
    height_scale=1.0,
    radius=1.0,
    alpha=2.0,
    timestep=0.01,
    mass=1.0,
)

sim.run()
updated_strands = sim.get_strands()

# Visualize the updated strands
updated_strands = np.array(updated_strands)
plot_cls(updated_strands)